# Fine-Tuning Orchestrator

Notebook ini mengatur pelatihan model Bi-Encoder dan Cross-Encoder untuk CV Matching System.

## Struktur Direktori
- `data/training/`: Dataset CSV (Triplet & Pairs)
- `training/scripts/`: Script pelatihan Python
- `models/`: Output model terlatih

## Langkah Persiapan
1. Pastikan dataset CSV tersedia di `data/training/`
2. Install dependencies: `pip install -r backend/requirements.txt`

In [5]:
# Setup Paths
import os
import sys

# Add training scripts to path
sys.path.append('training/scripts')

# Define paths
DATA_DIR = '/home/kuliah/intern/dicoding/caps-final/data/training'
MODEL_DIR = '/home/kuliah/intern/dicoding/caps-final/models'
SCRIPT_DIR = '/home/kuliah/intern/dicoding/caps-final/training/scripts'
SKILLS_DIR = '/home/kuliah/intern/dicoding/caps-final/backend/app/core/skills'
DATASET_SCRIPT = '/home/kuliah/intern/dicoding/caps-final/training/scripts/generate_dataset.py'
DATASET_TEMPLATE = '/home/kuliah/intern/dicoding/caps-final/training/templates'

# Check directory structure
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Data directory: {os.path.abspath(DATA_DIR)}")
print(f"Model directory: {os.path.abspath(MODEL_DIR)}")
print(f"Script directory: {os.path.abspath(SCRIPT_DIR)}")

Data directory: /home/kuliah/intern/dicoding/caps-final/data/training
Model directory: /home/kuliah/intern/dicoding/caps-final/models
Script directory: /home/kuliah/intern/dicoding/caps-final/training/scripts


## Phase 1: Data Preparation

Dataset yang diperlukan:
1. `bi_encoder_train.csv` (Triplet: anchor, positive, negative)

Anda dapat menggunakan data sintetis untuk testing atau siapkan data real dari CV dan JD Anda.

In [6]:
# Generate synthetic data using script
import subprocess

def generate_synthetic_data(num_triplets=2000):
    print("Generating dataset using training/scripts/generate_dataset.py...")
    cmd = [
        'python', DATASET_SCRIPT,
        '--skills_dir', SKILLS_DIR,
        '--templates_dir', DATASET_TEMPLATE,
        '--output_dir', DATA_DIR,
        '--num_triplets', str(num_triplets)
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(f"Error: {result.stderr}")
    except Exception as e:
        print(f"Exception during generation: {e}")

# Run synthetic data generation
generate_synthetic_data()

# Check existing data
if os.path.exists(f'{DATA_DIR}/bi_encoder_train.csv'):
    print("Bi-Encoder training data found.")
else:
    print("Bi-Encoder training data NOT found.")

Generating dataset using training/scripts/generate_dataset.py...
Generated and saved Bi-Encoder triplet dataset to: /home/kuliah/intern/dicoding/caps-final/data/training/bi_encoder_train.csv

Bi-Encoder training data found.


## Phase 2: Bi-Encoder Training

Melatih model Bi-Encoder menggunakan data triplet untuk menghasilkan embedding yang akurat.

In [7]:
import sys
import subprocess

def train_bi_encoder():
    train_csv = f'{DATA_DIR}/bi_encoder_train.csv'
    output_path = f'{MODEL_DIR}/bi-encoder-cv-matcher'
    
    if not os.path.exists(train_csv):
        print(f"Error: Training data not found at {train_csv}")
        return
    
    print(f"Training Bi-Encoder...")
    print(f"Input: {train_csv}")
    print(f"Output: {output_path}")
    
    # Run training script
    cmd = [
        sys.executable, f'{SCRIPT_DIR}/train_bi_encoder.py',
        '--train_csv', train_csv,
        '--output_path', output_path,
        '--epochs', '10',
        '--batch_size', '16'
    ]
    
    try:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end='')
        process.wait()
        if process.returncode != 0:
            print(f"Error occurred during training. Return code: {process.returncode}")
        else:
            print("Bi-Encoder training completed successfully!")
    except Exception as e:
        print(f"Exception during training: {e}")

# Uncomment to run training
train_bi_encoder()


Training Bi-Encoder...
Input: /home/kuliah/intern/dicoding/caps-final/data/training/bi_encoder_train.csv
Output: /home/kuliah/intern/dicoding/caps-final/models/bi-encoder-cv-matcher
Loading base model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Loading training data from: /home/kuliah/intern/dicoding/caps-final/data/training/bi_encoder_train.csv
No separate evaluation CSV provided. Splitting training data 90/10...
Training samples: 1800, Evaluation samples: 200
Setting up training configurations...
Setting up evaluation metrics...
Starting Bi-Encoder training with optimization...
Trying training with batch_size: 16, gradient_accumulation_steps: 1

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]
                                                                     

  1%|          | 10/1130 [00:04<08:05,  2.31it/s]
                                                 
{'loss': 3.7584, 'grad_norm': 50.82087707519531, 'learning_rate': 1.415929203539823e-0

## Phase 3: Model Verification

Verifikasi model yang telah dilatih tersimpan dengan benar.

In [8]:
from sentence_transformers import SentenceTransformer

def verify_models():
    # Check Bi-Encoder
    bi_encoder_path = f'{MODEL_DIR}/bi-encoder-cv-matcher'
    if os.path.exists(bi_encoder_path):
        try:
            model = SentenceTransformer(bi_encoder_path)
            print(f"✓ Bi-Encoder loaded successfully from {bi_encoder_path}")
            # Test inference
            test_embedding = model.encode("Test sentence")
            print(f"  Embedding shape: {test_embedding.shape}")
        except Exception as e:
            print(f"✗ Error loading Bi-Encoder: {e}")
    else:
        print(f"✗ Bi-Encoder model not found at {bi_encoder_path}")

verify_models()


✓ Bi-Encoder loaded successfully from /home/kuliah/intern/dicoding/caps-final/models/bi-encoder-cv-matcher
  Embedding shape: (384,)
